# Cutout Size Determination by detector

Cutout size varies with max dy, which varies by detector. This notebook can read a conf file and determine the max dy. Specifically, the scipy.optimize function minimize returns the minimum -dy (negative dy) and it's location (the negative of the minimum -dy is the maximum). The absolute value of this is the minimum size cutout.

1) Put the conf file in.
2) Run the notebook.
3) Note the value returned at 'x:'; this is the location of the maximum.
4) Note the value returned at 'func:'; this times -1 is the maximum value.
5) Test somewhere that setting size equal to this func value works, or one greater. This test is in case scipy got stuck in a local max/min.
6) Track the detector value.

det1: 77  - tested \
det2: 80 \
det3: 88 \
det4: 180 \
det5: 195 \
det6: 209 \
det7: 279 \
det8: 302 \
det9: 328 \
det10: 98 \
det11: 101 \
det12: 110 \
det13: 203 \
det14: 217 \
det15: 232 \
det16: 303 \
det17: 326 \
det18: 353

In [1]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize, Bounds

import os
os.chdir("/Users/keith/astr/research_astr/roman_configs")

In [11]:
# Read Config File
from collections import OrderedDict
config_file = "Roman.det1.07242020.conf"
config_file = "/Users/keith/astr/research_astr/grism_sim/data/Roman.det1.03192025_rotAonly.conf"

conf = OrderedDict()
lines = open(config_file).readlines()

for line in lines:
    if (line.startswith('#')) | (line.strip() == '') | ('"' in line):
                continue

    if line.startswith("DYDX_A_"):
        spl = line.split(';')[0].split('#')[0].split()
        param = spl[0]
        value = np.cast[float](spl[1:])
        conf[param] = value

b = {}
for n, dydx in enumerate(conf.keys()):
    for m, coef in enumerate(conf[dydx]):
        b[f"{n},{m}"] = coef

In [12]:
# Define functions
a_0 = lambda i, j: b["0,0"] + b["0,1"]*i + b["0,2"]*j + b["0,3"]*i**2 + b["0,4"]*i*j + b["0,5"]*j**2
a_1 = lambda i, j: b["1,0"] + b["1,1"]*i + b["1,2"]*j + b["1,3"]*i**2 + b["1,4"]*i*j + b["1,5"]*j**2
a_2 = lambda i, j: b["2,0"] + b["2,1"]*i + b["2,2"]*j + b["2,3"]*i**2 + b["2,4"]*i*j + b["2,5"]*j**2

# args[2] is dx; this is provided and probed by scipy.optimize
dy = lambda args: (a_0(args[0],args[1]) + a_1(args[0],args[1])*args[2] + a_2(args[0],args[1])*args[2]**2)
dy_negated = lambda args: -(a_0(args[0],args[1]) + a_1(args[0],args[1])*args[2] + a_2(args[0],args[1])*args[2]**2)

In [15]:
# Define Bounds
bounds = Bounds([-1000,-1000,-800],[6800,6800,800])

# Optimize; take the most extreme displacement, up or down
max(
abs(minimize(dy, x0=[2044,2044,0], bounds=bounds).fun),
abs(minimize(dy_negated, x0=[2044,2044,0], bounds=bounds).fun)
)

17.7298229009856

In [16]:
foo = minimize(dy, x0=[2044,2044,0], bounds=bounds)
bar = minimize(dy_negated, x0=[2044,2044,0], bounds=bounds)

print(foo.fun, foo.x)
print(bar.fun, bar.x)

-17.7298229009856 [ 6800. -1000.   800.]
-16.725135866054398 [ 6800. -1000.  -800.]


In [5]:
import grism_dispersion

In [6]:
def test_foot(xpix,ypix,min_pix=0,max_pix=4088,det='1'):
    conf = grism_dispersion.aXeConf(conf_file="/Users/keith/astr/research_astr/roman_configs/Roman.det18.07242020.conf")
    dy,lam = conf.get_beam_trace(x=xpix, y=ypix, dx=all_dx, beam='A')
    sel_lam = lam > max_lam_4foot
    sel_lam |= lam < min_lam_4foot
    dx_sel = all_dx[~sel_lam]
    dy_sel = dy[~sel_lam]
    return max(dy_sel)
    if xpix+dx_sel[0] >= 0 and xpix+dx_sel[-1] < 4088 and ypix+dy_sel[0] >= 0 and ypix+dy_sel[-1] < 4088 and ypix+dy_sel[0] < 4088 and ypix+dy_sel[-1] >= 0:
        return 1
    else:
        return 0

In [7]:
min_lam_4foot = 10000
max_lam_4foot = 19300
all_dx = np.arange(-400,600)

test_foot(100, 0)

-221.7864770396992

In [8]:
full_test_foot = []
for x in np.arange(1, 4088, 10):
    for y in np.arange(1, 4088, 10):
        full_test_foot.append(test_foot(x, y))

In [9]:
max(full_test_foot)

-69.14968976415955

In [10]:
min(full_test_foot)

-268.8730424420949